### Cell 1: Configuration & Widgets

In [0]:
# Widgets
dbutils.widgets.text("input_folder", "/Volumes/1_inland/sectra/vone/Hackathon3/", "Input DICOM Folder")
dbutils.widgets.text("ocr_log_table", "8_dev.pacs.image_ocr_text", "OCR Log Table")
dbutils.widgets.text("preprocessing", "True", "OCR Preprocessing (True/False)")
dbutils.widgets.text("min_text_length", "3", "Min OCR Text Length")
dbutils.widgets.text("padding", "10", "Redaction Padding (px)")
dbutils.widgets.text("batch_size", "50", "Images per worker batch")
dbutils.widgets.text("max_workers", "8", "Max GPU workers (for autoscaling clusters)")

# Environment variables — must be set before PaddleOCR import
import os
os.environ["FLAGS_fraction_of_gpu_memory_to_use"] = "0.5"
os.environ.setdefault("PADDLE_PDX_ENABLE_MKLDNN_BYDEFAULT", "False")
os.environ.setdefault("PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK", "True")

In [0]:
%pip install SimpleITK

In [0]:
# Restart Python so env vars take effect before paddle imports
dbutils.library.restartPython()

### Cell 2: Imports & GPU Configuration

In [0]:
import os
import re
import math
import calendar
import datetime
import json
from collections import defaultdict
from pathlib import Path
from typing import Iterator

import numpy as np
import cv2
import SimpleITK as sitk
import pandas as pd

from pyspark.sql.functions import col, lit, current_timestamp, collect_set, when, collect_list, struct
from pyspark.sql.types import (
    StructType, StructField, StringType, BooleanType, IntegerType, TimestampType
)

# Read widget values
INPUT_FOLDER = dbutils.widgets.get("input_folder")
OCR_LOG_TABLE = dbutils.widgets.get("ocr_log_table")
PREPROCESSING = dbutils.widgets.get("preprocessing").strip().lower() == "true"
MIN_TEXT_LENGTH = int(dbutils.widgets.get("min_text_length"))
PADDING = int(dbutils.widgets.get("padding"))
BATCH_SIZE = int(dbutils.widgets.get("batch_size"))
MAX_WORKERS = int(dbutils.widgets.get("max_workers"))

# Derive output folder with timestamp
_run_ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_FOLDER = f"/Volumes/8_dev/pacs/anon_images/{_run_ts}/"

print(f"Input folder:  {INPUT_FOLDER}")
print(f"Output folder: {OUTPUT_FOLDER}")
print(f"OCR log table: {OCR_LOG_TABLE}")
print(f"Preprocessing: {PREPROCESSING}")
print(f"Min text len:  {MIN_TEXT_LENGTH}")
print(f"Padding:       {PADDING}")
print(f"Batch size:    {BATCH_SIZE}")
print(f"Max workers:   {MAX_WORKERS}")

# ── GPU-aware configuration ───────────────────────────────────────────────────
# Ensure each Spark task gets exactly 1 GPU — prevents multiple PaddleOCR
# instances from competing for the same GPU memory.
spark.conf.set("spark.sql.execution.arrow.maxRecordsPerBatch", "50")
spark.conf.set("spark.task.resource.gpu.amount", "1")
spark.conf.set("spark.rapids.sql.enabled", "false")

# Use max_workers widget for partitioning — works with autoscaling clusters.
# Over-partitioning is fine: Spark queues excess partitions and runs them as
# GPUs become available. This avoids the problem of snapshotting executor count
# at startup when the cluster may still be scaling up.
_gpus_per_executor = int(spark.conf.get("spark.executor.resource.gpu.amount", "1"))
num_gpu_partitions = MAX_WORKERS * _gpus_per_executor

# Current executors (for info only — partitioning uses max_workers)
_current_executors = max(1, sc._jsc.sc().getExecutorMemoryStatus().size() - 1)

print(f"Current executors: {_current_executors}")
print(f"GPUs per executor: {_gpus_per_executor}")
print(f"GPU partitions:    {num_gpu_partitions} (based on max_workers={MAX_WORKERS})")

### Cell 3: Helper functions

In [0]:
# ── Image Helpers ─────────────────────────────────────────────────────────────

def _normalize_to_uint8(image_array):
    """Normalize any numeric array to uint8 0-255 for OCR input."""
    if image_array.ndim == 3:
        image_array = np.squeeze(image_array)
    if image_array.ndim == 3:
        image_array = image_array[0]
    img = image_array.astype(np.float32)
    img_min, img_max = img.min(), img.max()
    if img_max - img_min > 0:
        img = (img - img_min) / (img_max - img_min) * 255
    return img.astype(np.uint8)


def _preprocess(img_uint8):
    """Invert + CLAHE for white-on-dark ultrasound text."""
    inverted = cv2.bitwise_not(img_uint8)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    return clahe.apply(inverted)


def _polygon_to_bbox(polygon):
    """Convert PaddleOCR quadrilateral to {x, y, width, height}."""
    xs = [pt[0] for pt in polygon]
    ys = [pt[1] for pt in polygon]
    x_min, x_max = float(min(xs)), float(max(xs))
    y_min, y_max = float(min(ys)), float(max(ys))
    return {"x": x_min, "y": y_min, "width": x_max - x_min, "height": y_max - y_min}


# ── OCR functions ─────────────────────────────────────────────────────────────

def run_ocr_structured(ocr_engine, numpy_array, preprocessing=True):
    """Full OCR: detection + recognition. Returns dict with text and bounding boxes."""
    img = _normalize_to_uint8(numpy_array)
    if preprocessing:
        img = _preprocess(img)
    if img.ndim == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)

    result = ocr_engine.ocr(img, cls=False)

    ocr_results = {}
    line_index = 0
    if result and result[0]:
        for detection in result[0]:
            polygon = detection[0]
            text = detection[1][0]
            ocr_results[f"line_{line_index}"] = {
                "text": text,
                "bounding_box": _polygon_to_bbox(polygon),
                "has_pii": False,
                "pii_details": [],
            }
            line_index += 1
    return ocr_results


def run_detection_only(ocr_engine, numpy_array, preprocessing=True):
    """Detection only: returns bounding boxes for all text regions (blanket mode)."""
    img = _normalize_to_uint8(numpy_array)
    if preprocessing:
        img = _preprocess(img)
    if img.ndim == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)

    result = ocr_engine.ocr(img, rec=False, cls=False)

    ocr_results = {}
    line_index = 0
    if result and result[0]:
        for polygon in result[0]:
            ocr_results[f"line_{line_index}"] = {
                "text": "",
                "bounding_box": _polygon_to_bbox(polygon),
                "has_pii": True,
                "pii_details": [],
            }
            line_index += 1
    return ocr_results


# ── Text Filter ───────────────────────────────────────────────────────────────

def filter_ocr_results(ocr_results, min_length=2, min_alpha_ratio=0.0):
    """Filter OCR results, marking short/non-text detections as non-PII."""
    filtered = {}
    for line_id, data in ocr_results.items():
        text = data["text"].strip()
        entry = {
            "text": data["text"],
            "bounding_box": data["bounding_box"],
            "pii_details": data.get("pii_details", []),
            "filtered": False,
            "filter_reason": None,
        }

        if len(text) < min_length:
            entry["has_pii"] = False
            entry["filtered"] = True
            entry["filter_reason"] = f"too_short (len={len(text)} < {min_length})"
            filtered[line_id] = entry
            continue

        if min_alpha_ratio > 0:
            total = len(text.replace(" ", ""))
            alpha = sum(1 for c in text if c.isalpha())
            ratio = alpha / max(total, 1)
            if ratio < min_alpha_ratio:
                entry["has_pii"] = False
                entry["filtered"] = True
                entry["filter_reason"] = f"low_alpha (ratio={ratio:.0%} < {min_alpha_ratio:.0%})"
                filtered[line_id] = entry
                continue

        entry["has_pii"] = True
        filtered[line_id] = entry

    return filtered


# ── PII Detection ─────────────────────────────────────────────────────────────

_PII_PATTERNS = [
    ("MRN", re.compile(r"\bMRN[:\s]*\d{6,10}\b", re.IGNORECASE)),
    ("ACCESSION", re.compile(r"\bACC(?:ESSION)?[:\s#]*\w{4,15}\b", re.IGNORECASE)),
    ("PATIENT_ID", re.compile(r"\bPID[:\s]*\d{4,12}\b", re.IGNORECASE)),
    ("GENDER", re.compile(r"\b(?:Male|Female|MALE|FEMALE)\b")),
    ("DATE", re.compile(
        r"\b\d{1,2}[/\-\.]\d{1,2}[/\-\.]\d{2,4}\b"
        r"|\b\d{4}[/\-\.]\d{1,2}[/\-\.]\d{1,2}\b"
        r"|\b\d{8}\b"
    )),
]

_ULTRASOUND_ALLOWLIST = {
    "sos", "map", "b/0", "b/1", "b/2", "m/0", "m/1",
    "logiq", "voluson", "vivid",
    "epiq", "affiniti", "iu22", "cx50",
    "acuson", "sequoia", "juniper", "helx",
    "aplio", "xario", "toshiba", "canon",
    "samsung", "rs85", "hs70", "hs60",
    "mindray", "resona", "dc-80", "dc-70",
    "arietta", "lisendo", "hitachi", "fujifilm",
    "frq", "freq", "gain", "tis", "tib", "tic", "tls",
    "prf", "wf", "thi", "cpa", "cpd", "pwr", "fps",
    "cw", "pw", "sv", "cf", "cci",
    "breast", "abdomen", "thyroid", "liver", "kidney", "ob",
    "cardiac", "vascular", "msk", "carotid", "renal",
    "dist", "area", "vol", "circ", "vel", "ari", "eri",
}


def is_valid_nhs_number(nhs_number):
    """Validate NHS number using checksum algorithm."""
    if not isinstance(nhs_number, str):
        return False
    nhs_digits = re.sub(r'[\s-]', '', nhs_number)
    if not nhs_digits.isdigit() or len(nhs_digits) != 10:
        return False
    weights = [10, 9, 8, 7, 6, 5, 4, 3, 2]
    total = sum(int(digit) * weight for digit, weight in zip(nhs_digits[:9], weights))
    remainder = total % 11
    check_digit = 11 - remainder
    if check_digit == 11:
        check_digit = 0
    elif check_digit == 10:
        return False
    return check_digit == int(nhs_digits[9])


def _build_dob_patterns(dob_dt):
    """Build regex patterns for common DOB renderings."""
    if dob_dt is None:
        return []

    day = dob_dt.day
    month = dob_dt.month
    year = dob_dt.year

    d = str(day)
    dd = f"{day:02d}"
    m = str(month)
    mm = f"{month:02d}"
    yyyy = str(year)
    yy = f"{year % 100:02d}"

    month_full = calendar.month_name[month]
    month_abbr = calendar.month_abbr[month]
    months_regex = f"(?:{re.escape(month_full)}|{re.escape(month_abbr)})"
    sep = r"[.\-/\s]"
    ord_suffix = r"(?:st|nd|rd|th)?"

    patterns = [
        fr"(?<!\d){dd}{sep}{mm}{sep}{yyyy}(?!\d)",
        fr"(?<!\d){d}{sep}{m}{sep}{yyyy}(?!\d)",
        fr"(?<!\d){dd}{sep}{mm}{sep}{yy}(?!\d)",
        fr"(?<!\d){d}{sep}{m}{sep}{yy}(?!\d)",
        fr"(?<!\d){yyyy}{sep}{mm}{sep}{dd}(?!\d)",
        fr"(?<!\d){yyyy}{sep}{m}{sep}{d}(?!\d)",
        fr"(?<!\d){dd}{mm}{yyyy}(?!\d)",
        fr"(?<!\d){yyyy}{mm}{dd}(?!\d)",
        fr"(?<!\d){dd}{mm}{yy}(?!\d)",
        fr"\b{d}{ord_suffix}{sep}+{months_regex}{sep}+{yyyy}\b",
        fr"\b{dd}{ord_suffix}{sep}+{months_regex}{sep}+{yyyy}\b",
        fr"\b{months_regex}{sep}+{d}{ord_suffix}{sep}+{yyyy}\b",
        fr"\b{months_regex}{sep}+{dd}{ord_suffix}{sep}+{yyyy}\b",
        fr"\b{d}{ord_suffix}{sep}+{months_regex}{sep}+{yy}\b",
        fr"\b{dd}{ord_suffix}{sep}+{months_regex}{sep}+{yy}\b",
        fr"\b{months_regex}{sep}+{d}{ord_suffix}{sep}+{yy}\b",
        fr"\b{dd}{ord_suffix}{sep}+{months_regex}{sep}+{yy}\b",
    ]
    return patterns


def detect_pii_in_ocr(ocr_results, deny_list):
    """Classify OCR text lines as PII using deny-list matching + regex patterns."""
    deny_set = {s.strip().lower() for s in deny_list if s and s.strip()}

    pii_results = {}
    for line_id, data in ocr_results.items():
        text = data["text"]
        found_entities = []

        text_lower = text.strip().lower()
        for deny_term in deny_set:
            if len(deny_term) < 2:
                continue
            if deny_term in text_lower:
                found_entities.append({
                    "entity_text": text,
                    "type": "PHI_MATCH",
                    "source": "deny_list",
                    "matched_term": deny_term,
                })
                break

        for pattern_name, pattern in _PII_PATTERNS:
            for match in pattern.finditer(text):
                matched = match.group()
                if matched.strip().lower() in _ULTRASOUND_ALLOWLIST:
                    continue
                found_entities.append({
                    "entity_text": matched,
                    "type": pattern_name,
                    "source": "regex",
                })

        for m in re.finditer(r'(?<!\d)(\d[\s-]?){9}\d(?!\d)', text):
            raw = m.group()
            if is_valid_nhs_number(raw):
                found_entities.append({
                    "entity_text": raw,
                    "type": "NHS_NUMBER",
                    "source": "nhs_checksum",
                })

        pii_results[line_id] = {
            "text": text,
            "bounding_box": data["bounding_box"],
            "has_pii": len(found_entities) > 0,
            "pii_details": found_entities,
            "filtered": data.get("filtered", False),
            "filter_reason": data.get("filter_reason"),
        }

    return pii_results


# ── Redaction ─────────────────────────────────────────────────────────────────

def redact_pii_from_image(image_array, pii_report, padding=10):
    """Black-fill bounding boxes flagged as PII in the image array."""
    redacted = image_array.copy()
    img_h, img_w = redacted.shape[:2]
    fill_value = int(redacted.min())

    for item in pii_report.values():
        if not item.get("has_pii"):
            continue
        box = item["bounding_box"]
        x_start = int(max(0, math.floor(box["x"] - padding)))
        y_start = int(max(0, math.floor(box["y"] - padding)))
        x_end = int(min(img_w, math.ceil(box["x"] + box["width"] + padding)))
        y_end = int(min(img_h, math.ceil(box["y"] + box["height"] + padding)))
        redacted[y_start:y_end, x_start:x_end] = fill_value

    return redacted


# ── DICOM Metadata ────────────────────────────────────────────────────────────

_PII_DICOM_TAGS = [
    "0010|0010",  # PatientName
    "0010|0020",  # PatientID
    "0010|0030",  # PatientBirthDate
    "0010|0040",  # PatientSex
    "0010|1000",  # OtherPatientIDs
    "0010|1001",  # OtherPatientNames
    "0010|2160",  # EthnicGroup
    "0008|0050",  # AccessionNumber
    "0008|0080",  # InstitutionName
    "0008|0090",  # ReferringPhysicianName
    "0008|1050",  # PerformingPhysicianName
    "0008|0020",  # StudyDate
    "0008|0030",  # StudyTime
    "0008|0021",  # SeriesDate
    "0008|0031",  # SeriesTime
    "0008|0023",  # ContentDate
    "0008|0033",  # ContentTime
    "0008|0022",  # AcquisitionDate
    "0008|0032",  # AcquisitionTime
    "0008|002A",  # AcquisitionDateTime

]

_DENY_LIST_DICOM_TAGS = {
    "0010|0010": "PatientName",
    "0010|0020": "PatientID",
    "0010|0030": "PatientBirthDate",
    "0008|0080": "InstitutionName",
    "0008|0090": "ReferringPhysicianName",
    "0008|0050": "AccessionNumber",
}


def _extract_metadata_deny_list(sitk_reader):
    """Extract PII values from DICOM headers for use as deny-list terms.

    Accepts either a sitk.Image or a sitk.ImageFileReader (both support
    HasMetaDataKey / GetMetaData).
    """
    deny_list = []
    for tag in _DENY_LIST_DICOM_TAGS:
        if sitk_reader.HasMetaDataKey(tag):
            value = sitk_reader.GetMetaData(tag).strip()
            if value:
                deny_list.append(value)
                if "^" in value:
                    deny_list.append(value.replace("^", " "))
                    for part in value.split("^"):
                        part = part.strip()
                        if part and len(part) > 1:
                            deny_list.append(part)
    return deny_list


def strip_dicom_pii(sitk_image):
    """Remove PII tags from a SimpleITK image's DICOM metadata (in-place)."""
    for tag in _PII_DICOM_TAGS:
        if sitk_image.HasMetaDataKey(tag):
            sitk_image.SetMetaData(tag, "")


    # 2) Pattern-based removal for PatientGroup (0010,xxxx ranges)
    keys_to_strip = []

    for key in sitk_image.GetMetaDataKeys():
        try:
            g_str, e_str = key.split("|")
            group = int(g_str, 16)
            element = int(e_str, 16)
        except Exception:
            continue

        if group == 0x0010 and (
            0x1000 <= element <= 0x10FF or
            0x2100 <= element <= 0x21FF
        ):
            keys_to_strip.append(key)

    for key in keys_to_strip:
        sitk_image.SetMetaData(key, "")
        
    return sitk_image


def build_deny_list_from_phi(phi_dict, dicom_header_deny_list):
    """Build a combined deny-list from Millennium PHI + DICOM header values."""
    deny_list = list(dicom_header_deny_list)

    for name_list in [phi_dict.get("first_names", []),
                      phi_dict.get("middle_names", []),
                      phi_dict.get("last_names", [])]:
        for name in name_list:
            if name and len(name) > 1:
                deny_list.append(name)

    for alias in phi_dict.get("aliases", []):
        if alias and len(alias) > 2:
            deny_list.append(alias)

    dob = phi_dict.get("dob")
    if dob:
        dd = f"{dob.day:02d}"
        mm = f"{dob.month:02d}"
        yyyy = str(dob.year)
        deny_list.append(f"{dd}/{mm}/{yyyy}")
        deny_list.append(f"{dd}-{mm}-{yyyy}")
        deny_list.append(f"{dd}.{mm}.{yyyy}")
        deny_list.append(f"{yyyy}{mm}{dd}")
        deny_list.append(f"{dd}{mm}{yyyy}")

    for addr in phi_dict.get("addresses", []):
        for field in ["STREET_ADDR", "STREET_ADDR2", "STREET_ADDR3", "STREET_ADDR4",
                      "CITY", "COUNTY", "STATE", "COUNTRY", "ZIPCODE", "POSTAL_IDENTIFIER"]:
            val = addr.get(field) if isinstance(addr, dict) else getattr(addr, field, None)
            if val and len(str(val).strip()) > 2:
                deny_list.append(str(val).strip())

    return deny_list

### Cell 4: Discover Files

Uses `dbutils.fs.ls` recursively — fast metadata listing, no DICOM I/O.

In [0]:
def _list_files_recursive(path):
    """Recursively list files using dbutils.fs.ls (metadata only, no file reads)."""
    files = []
    try:
        items = dbutils.fs.ls(path)
    except Exception:
        return files
    for item in items:
        if item.isDir():
            files.extend(_list_files_recursive(item.path))
        else:
            files.append(item.path)
    return files


print(f"Scanning {INPUT_FOLDER} for files...")
_dicom_exts = {".dcm", ".mhd", ".nii", ".nrrd"}
all_files = _list_files_recursive(INPUT_FOLDER)

dicom_files = []
for fp in all_files:
    suffix = Path(fp).suffix.lower()
    if suffix in _dicom_exts or "." not in Path(fp).name:
        dicom_files.append(fp)

# Normalise dbfs:/Volumes → /Volumes
dicom_paths = []
for fp in dicom_files:
    if fp.startswith("dbfs:/Volumes"):
        dicom_paths.append(fp.replace("dbfs:", "", 1))
    elif fp.startswith("/Volumes"):
        dicom_paths.append(fp)
    else:
        dicom_paths.append(fp)

print(f"Found {len(dicom_paths)} DICOM files")

### Cell 5: Pass 1 — Distributed Metadata Extraction

Reads DICOM headers **without decoding pixel data** using
`sitk.ImageFileReader.ReadImageInformation()`. Returns accession number,
study UID, frame count, and DICOM-header deny-list terms per file.

Pass 1 is CPU-only work — use more partitions for parallelism.

In [0]:
_PASS1_SCHEMA = StructType([
    StructField("file_path", StringType(), False),
    StructField("accession_nbr", StringType(), True),
    StructField("study_uid", StringType(), True),
    StructField("patient_id_tag", StringType(), True),
    StructField("header_deny_json", StringType(), True),
    StructField("has_pixels", BooleanType(), False),
    StructField("num_frames", IntegerType(), True),
    StructField("error", StringType(), True),
])


def extract_headers(batch_iter: Iterator[pd.DataFrame]) -> Iterator[pd.DataFrame]:
    """Pass 1 worker: read DICOM metadata only (no pixel decode)."""
    import SimpleITK as sitk
    import json

    for batch_df in batch_iter:
        results = []
        for _, row in batch_df.iterrows():
            file_path = row["file_path"]
            try:
                reader = sitk.ImageFileReader()
                reader.SetFileName(file_path)
                reader.LoadPrivateTagsOn()
                reader.ReadImageInformation()

                acc = reader.GetMetaData("0008|0050").strip() if reader.HasMetaDataKey("0008|0050") else ""
                uid = reader.GetMetaData("0020|000d").strip() if reader.HasMetaDataKey("0020|000d") else ""
                pid = reader.GetMetaData("0010|0020").strip() if reader.HasMetaDataKey("0010|0020") else ""
                header_deny = _extract_metadata_deny_list(reader)

                # Check for pixel data via Rows tag (0028|0010)
                has_pixels = reader.HasMetaDataKey("0028|0010")

                # Frame count from NumberOfFrames tag (0028|0008), default 1
                num_frames = 1
                if reader.HasMetaDataKey("0028|0008"):
                    try:
                        num_frames = int(reader.GetMetaData("0028|0008").strip())
                    except (ValueError, TypeError):
                        num_frames = 1

                results.append({
                    "file_path": file_path,
                    "accession_nbr": acc or None,
                    "study_uid": uid or None,
                    "patient_id_tag": pid or None,
                    "header_deny_json": json.dumps(header_deny),
                    "has_pixels": has_pixels,
                    "num_frames": num_frames,
                    "error": None,
                })
            except Exception as e:
                results.append({
                    "file_path": file_path,
                    "accession_nbr": None,
                    "study_uid": None,
                    "patient_id_tag": None,
                    "header_deny_json": "[]",
                    "has_pixels": False,
                    "num_frames": 0,
                    "error": str(e),
                })

        yield pd.DataFrame(results) if results else pd.DataFrame(columns=_PASS1_SCHEMA.fieldNames())


# Build input DataFrame — use more partitions for CPU-only Pass 1
_p1_rows = [{"file_path": fp} for fp in dicom_paths]
_p1_input = spark.createDataFrame(_p1_rows, schema=StructType([StructField("file_path", StringType(), False)]))

_p1_partitions = max(num_gpu_partitions, sc.defaultParallelism)
_p1_input = _p1_input.repartition(_p1_partitions)

print(f"Pass 1: extracting DICOM headers across {_p1_partitions} partitions...")
headers_df = _p1_input.mapInPandas(extract_headers, schema=_PASS1_SCHEMA)
headers_df.cache()

total_files = headers_df.count()
readable_count = headers_df.filter(col("has_pixels") == True).filter(col("error").isNull()).count()
skipped_count = total_files - readable_count
multiframe_count = headers_df.filter(col("num_frames") > 1).count()
print(f"  {readable_count} readable, {skipped_count} skipped (no pixels or unreadable)")
print(f"  {multiframe_count} multi-frame DICOMs (will extract frame 0 only)")

### Cell 6: Patient Linking & PHI Gathering (Driver-side, Spark SQL)

Joins Pass 1 headers with `imaging_metadata` to link files to patients,
then gathers PHI for **only the matched patients** (not all 2M).

In [0]:
# Filter to readable files only
valid_headers_df = headers_df.filter(col("has_pixels") == True).filter(col("error").isNull())

# ── Link via accession number ─────────────────────────────────────────────────
imaging_meta = spark.table("4_prod.pacs.imaging_metadata")

acc_linked = (
    valid_headers_df.alias("h")
    .join(
        imaging_meta.filter(col("PersonId").isNotNull())
                    .select(col("AccessionNbr"), col("PersonId"))
                    .distinct()
                    .alias("m"),
        col("h.accession_nbr") == col("m.AccessionNbr"),
        "left",
    )
    .select(
        col("h.file_path"),
        col("h.accession_nbr"),
        col("h.study_uid"),
        col("h.patient_id_tag"),
        col("h.header_deny_json"),
        col("h.num_frames"),
        col("m.PersonId").alias("person_id"),
    )
)

# ── Fall back to study UID for unlinked files ─────────────────────────────────
unlinked = acc_linked.filter(col("person_id").isNull())
linked = acc_linked.filter(col("person_id").isNotNull())

uid_linked = (
    unlinked.drop("person_id").alias("u")
    .join(
        imaging_meta.filter(col("PersonId").isNotNull())
                    .select(col("ExaminationStudyUid"), col("PersonId"))
                    .distinct()
                    .alias("m2"),
        col("u.study_uid") == col("m2.ExaminationStudyUid"),
        "left",
    )
    .select(
        col("u.file_path"),
        col("u.accession_nbr"),
        col("u.study_uid"),
        col("u.patient_id_tag"),
        col("u.header_deny_json"),
        col("u.num_frames"),
        col("m2.PersonId").alias("person_id"),
    )
)

# Combine linked + uid-linked
all_files_df = linked.unionByName(uid_linked).cache()

linked_count = all_files_df.filter(col("person_id").isNotNull()).count()
unlinked_count = all_files_df.filter(col("person_id").isNull()).count()
print(f"  {linked_count} linked to patients, {unlinked_count} unlinked (will blanket-redact)")

# ── Gather PHI only for patients in this batch ────────────────────────────────
matched_pids = [
    row.person_id for row in
    all_files_df.filter(col("person_id").isNotNull())
               .select("person_id").distinct().collect()
]
print(f"  Unique patients in batch: {len(matched_pids)}")

phi_data = {}
if matched_pids:
    print("Gathering PHI for matched patients...")

    # Names
    names_df = (
        spark.table("4_prod.raw.mill_person_name")
        .filter(col("PERSON_ID").isin(matched_pids))
        .groupBy("PERSON_ID")
        .agg(
            collect_set(when(col("NAME_FIRST").isNotNull(), col("NAME_FIRST"))).alias("first_names"),
            collect_set(when(col("NAME_MIDDLE").isNotNull(), col("NAME_MIDDLE"))).alias("middle_names"),
            collect_set(when(col("NAME_LAST").isNotNull(), col("NAME_LAST"))).alias("last_names"),
        )
        .collect()
    )
    for row in names_df:
        phi_data[row.PERSON_ID] = {
            "first_names": [n for n in (row.first_names or []) if n and n.strip()],
            "middle_names": [n for n in (row.middle_names or []) if n and n.strip()],
            "last_names": [n for n in (row.last_names or []) if n and n.strip()],
            "dob": None, "addresses": [], "aliases": [],
        }

    for pid in matched_pids:
        if pid not in phi_data:
            phi_data[pid] = {
                "first_names": [], "middle_names": [], "last_names": [],
                "dob": None, "addresses": [], "aliases": [],
            }

    # DOB
    dob_df = (
        spark.table("4_prod.raw.mill_person")
        .filter(col("PERSON_ID").isin(matched_pids))
        .select("PERSON_ID", "BIRTH_DT_TM")
        .collect()
    )
    for row in dob_df:
        if row.PERSON_ID in phi_data and row.BIRTH_DT_TM:
            phi_data[row.PERSON_ID]["dob"] = row.BIRTH_DT_TM

    # Addresses — convert Row objects to plain dicts for serialization
    addr_df = (
        spark.table("4_prod.raw.mill_address")
        .filter(col("PARENT_ENTITY_NAME") == "PERSON")
        .filter(col("PARENT_ENTITY_ID").isin(matched_pids))
        .filter(col("ACTIVE_IND") == 1)
        .select(
            col("PARENT_ENTITY_ID").alias("PERSON_ID"),
            struct(
                "STREET_ADDR", "STREET_ADDR2", "STREET_ADDR3", "STREET_ADDR4",
                "CITY", "COUNTY", "STATE", "COUNTRY", "ZIPCODE", "POSTAL_IDENTIFIER"
            ).alias("address"),
        )
        .groupBy("PERSON_ID")
        .agg(collect_list("address").alias("addresses"))
        .collect()
    )
    for row in addr_df:
        if row.PERSON_ID in phi_data:
            phi_data[row.PERSON_ID]["addresses"] = [
                addr.asDict() for addr in (row.addresses or [])
            ]

    # Aliases
    alias_df = (
        spark.table("4_prod.raw.mill_person_alias")
        .filter(col("PERSON_ID").isin(matched_pids))
        .filter(col("ACTIVE_IND") == 1)
        .filter(col("ALIAS").isNotNull())
        .filter(col("ALIAS") != "")
        .groupBy("PERSON_ID")
        .agg(collect_set("ALIAS").alias("aliases"))
        .collect()
    )
    for row in alias_df:
        if row.PERSON_ID in phi_data:
            phi_data[row.PERSON_ID]["aliases"] = [
                a for a in (row.aliases or []) if a and a.strip()
            ]

    print(f"  PHI gathered for {len(phi_data)} patients")

# Pre-build DOB patterns per patient
phi_dob_patterns = {}
for pid, phi in phi_data.items():
    dob = phi.get("dob")
    phi_dob_patterns[pid] = _build_dob_patterns(dob) if dob else []

### Cell 7: Build Pass 2 Work DataFrame

Merges DICOM-header deny lists with PHI-based deny lists into a single
JSON column per file, ready for workers. Includes `num_frames` so workers
can extract only frame 0 from multi-frame DICOMs.

In [0]:
# Collect file→person_id mapping so we can assemble deny lists
file_info = all_files_df.select(
    "file_path", "accession_nbr", "person_id", "header_deny_json", "num_frames"
).collect()

work_rows = []
for row in file_info:
    person_id = row.person_id
    header_deny = json.loads(row.header_deny_json)

    if False: #person_id is not None and person_id in phi_data:
        deny_list = build_deny_list_from_phi(phi_data[person_id], header_deny)
        dob_patterns = phi_dob_patterns.get(person_id, [])
        mode = "selective"
    else:
        deny_list = header_deny
        dob_patterns = []
        mode = "blanket"

    work_rows.append({
        "file_path": row.file_path,
        "deny_list_json": json.dumps(deny_list),
        "dob_patterns_json": json.dumps(dob_patterns),
        "person_id": str(person_id) if person_id else None,
        "mode": mode,
        "accession_nbr": row.accession_nbr or "",
        "num_frames": row.num_frames or 1,
    })

work_schema = StructType([
    StructField("file_path", StringType(), False),
    StructField("deny_list_json", StringType(), False),
    StructField("dob_patterns_json", StringType(), False),
    StructField("person_id", StringType(), True),
    StructField("mode", StringType(), False),
    StructField("accession_nbr", StringType(), True),
    StructField("num_frames", IntegerType(), False),
])

work_df = spark.createDataFrame(work_rows, schema=work_schema)
work_df = work_df.repartition(num_gpu_partitions)

print(f"Pass 2 work DataFrame: {len(work_rows)} files across {num_gpu_partitions} GPU partitions")
print(f"  Selective (linked): {sum(1 for r in work_rows if r['mode'] == 'selective')}")
print(f"  Blanket (unlinked): {sum(1 for r in work_rows if r['mode'] == 'blanket')}")
print(f"  Multi-frame:        {sum(1 for r in work_rows if r['num_frames'] > 1)}")

In [0]:
import os, glob, shutil

PADDLE_MODEL_VOL = "/Volumes/8_dev/pacs/anon_images/_paddle_models/en_v1"

def _stage_paddle_models():
    from paddleocr import PaddleOCR
    _ = PaddleOCR(lang="en", use_gpu=False, use_angle_cls=False, show_log=False)  # warms ~/.paddleocr
    cache_root = os.path.expanduser("~/.paddleocr/whl")
    dirs = {}
    for kind in ("det", "rec", "cls"):
        hits = glob.glob(os.path.join(cache_root, kind, "**", "inference.pdmodel"), recursive=True)
        if not hits:
            raise RuntimeError(f"No staged {kind} model under {cache_root}/{kind}")
        src_dir, dst_dir = os.path.dirname(hits[0]), os.path.join(PADDLE_MODEL_VOL, kind)
        os.makedirs(dst_dir, exist_ok=True)
        for f in os.listdir(src_dir):
            shutil.copy2(os.path.join(src_dir, f), os.path.join(dst_dir, f))
        dirs[kind] = dst_dir
    return dirs

if all(os.path.exists(os.path.join(PADDLE_MODEL_VOL, k, "inference.pdmodel")) for k in ("det", "rec", "cls")):
    PADDLE_DIRS = {k: os.path.join(PADDLE_MODEL_VOL, k) for k in ("det", "rec", "cls")}
    print("PaddleOCR models already staged.")
else:
    print("Staging PaddleOCR models to Volume (one-time)...")
    PADDLE_DIRS = _stage_paddle_models()

print("PaddleOCR model dirs:", PADDLE_DIRS)
_bc_paddle_dirs = sc.broadcast(PADDLE_DIRS)


### Cell 8: Pass 2 — Distributed OCR + Redaction (`mapInPandas`)

Partitioned by max GPU count. Each task gets exactly 1 GPU via
`spark.task.resource.gpu.amount = 1`. Spark queues excess partitions and
runs them as GPUs become available during autoscaling.

Multi-frame DICOMs: extracts only frame 0 to avoid loading GBs of pixel
data (e.g. a 2992-frame DICOM is ~750MB raw, ~3GB as float32).

In [0]:
_OCR_LOG_SCHEMA = StructType([
    StructField("source_file", StringType(), True),
    StructField("accession_nbr", StringType(), True),
    StructField("person_id", StringType(), True),
    StructField("line_index", IntegerType(), True),
    StructField("text", StringType(), True),
    StructField("bounding_box", StringType(), True),
    StructField("has_pii", BooleanType(), True),
    StructField("action", StringType(), True),
    StructField("pii_details", StringType(), True),
    StructField("filter_reason", StringType(), True),
    StructField("processed_at", StringType(), True),
    StructField("output_path", StringType(), True),
    StructField("error", StringType(), True),
])

_bc_preprocessing = sc.broadcast(PREPROCESSING)
_bc_min_text_length = sc.broadcast(MIN_TEXT_LENGTH)
_bc_padding = sc.broadcast(PADDING)
_bc_output_folder = sc.broadcast(OUTPUT_FOLDER)
_bc_input_folder = sc.broadcast(INPUT_FOLDER)


def process_batch(batch_iter: Iterator[pd.DataFrame]) -> Iterator[pd.DataFrame]:
    """Pass 2 worker: read DICOM pixels → OCR → PII detect → redact → save.

    For multi-frame DICOMs, uses sitk.Extract to read only frame 0 — avoids
    loading the entire volume into memory.
    """
    import os
    import re
    import math
    import json
    import datetime
    import gc
    import numpy as np
    import cv2
    import SimpleITK as sitk
    from pathlib import Path

    os.environ["FLAGS_fraction_of_gpu_memory_to_use"] = "0.5"

    from paddleocr import PaddleOCR

    paddle_dirs = _bc_paddle_dirs.value
    ocr_engine = PaddleOCR(
          lang="en",
          use_gpu=True,
          use_angle_cls=False,
          det_limit_side_len=960,
          show_log=False,
          det_model_dir=paddle_dirs["det"],
          rec_model_dir=paddle_dirs["rec"],
          cls_model_dir=paddle_dirs["cls"],
      )


    preprocessing = _bc_preprocessing.value
    min_text_length = _bc_min_text_length.value
    padding = _bc_padding.value
    output_folder = _bc_output_folder.value
    input_folder = _bc_input_folder.value

    processed_at = datetime.datetime.now().isoformat(timespec="seconds")

    def _ensure_dir(path):
        try:
            os.makedirs(path, exist_ok=True)
        except OSError:
            pass

    def _extract_first_frame(image, num_frames):
        """Extract first frame from a SimpleITK image, memory-efficiently.

        For multi-frame DICOMs, uses sitk.Extract to pull only frame 0
        instead of converting the entire volume to numpy first.
        """
        size = list(image.GetSize())
        if num_frames > 1 and len(size) == 3 and size[2] > 1:
            # Extract a single 2D slice (frame 0) from the 3D volume
            extract_size = [size[0], size[1], 0]  # 0 collapses that dimension
            extract_index = [0, 0, 0]
            frame_image = sitk.Extract(image, extract_size, extract_index)
            return sitk.GetArrayFromImage(frame_image)
        else:
            image_array = sitk.GetArrayFromImage(image)
            if image_array.ndim == 3:
                frame = image_array[0].copy()
                del image_array
                return frame
            return image_array

    for batch_df in batch_iter:
        results = []

        for _, row in batch_df.iterrows():
            file_path = row["file_path"]
            deny_list = json.loads(row["deny_list_json"])
            dob_patterns = json.loads(row["dob_patterns_json"])
            person_id = row["person_id"]
            mode = row["mode"]
            accession_nbr = row["accession_nbr"] or ""
            num_frames = int(row["num_frames"]) if row["num_frames"] else 1
            rel_path = os.path.relpath(file_path, input_folder)

            try:
                image = sitk.ReadImage(file_path)
                target_slice = _extract_first_frame(image, num_frames)

                if mode == "selective":
                    ocr_results = run_ocr_structured(ocr_engine, target_slice, preprocessing=preprocessing)
                    filtered = filter_ocr_results(ocr_results, min_length=min_text_length)
                    candidates = {k: v for k, v in filtered.items() if not v.get("filtered")}
                    pii_detected = detect_pii_in_ocr(candidates, deny_list)

                    if dob_patterns:
                        for line_id, data in pii_detected.items():
                            if not data["has_pii"]:
                                for pat in dob_patterns:
                                    if re.search(pat, data["text"], re.IGNORECASE):
                                        data["has_pii"] = True
                                        data["pii_details"].append({
                                            "entity_text": data["text"],
                                            "type": "DOB_PATTERN",
                                            "source": "dob_regex",
                                        })
                                        break

                    pii_report = {**filtered}
                    pii_report.update(pii_detected)
                else:
                    ocr_results = run_detection_only(ocr_engine, target_slice, preprocessing=preprocessing)
                    pii_report = filter_ocr_results(ocr_results, min_length=0)

                redacted_slice = redact_pii_from_image(target_slice, pii_report, padding=padding)
                del target_slice

                # Build output DICOM — always write as single-frame
                redacted_sitk = sitk.GetImageFromArray(redacted_slice)
                del redacted_slice

                # Copy spatial info from original (or first-frame extraction)
                # For multi-frame, we write a single-frame output
                if num_frames > 1 and len(image.GetSize()) == 3:
                    # Extract spatial metadata from frame 0
                    extract_size = [image.GetSize()[0], image.GetSize()[1], 0]
                    frame_ref = sitk.Extract(image, extract_size, [0, 0, 0])
                    redacted_sitk.CopyInformation(frame_ref)
                    del frame_ref
                else:
                    if redacted_sitk.GetDimension() == image.GetDimension():
                        redacted_sitk.CopyInformation(image)

                # Copy all metadata then strip PII tags
                for key in image.GetMetaDataKeys():
                    redacted_sitk.SetMetaData(key, image.GetMetaData(key))
                # Update frame count to 1 since we only write frame 0
                if redacted_sitk.HasMetaDataKey("0028|0008"):
                    redacted_sitk.SetMetaData("0028|0008", "1")
                strip_dicom_pii(redacted_sitk)

                del image
                gc.collect()

                # Save output (preserve subdirectory structure)
                rel_dir = os.path.dirname(rel_path)
                base_name = Path(rel_path).stem
                out_subdir = os.path.join(output_folder, rel_dir)
                _ensure_dir(out_subdir)
                dcm_out = os.path.join(out_subdir, f"{base_name}_anon.dcm")
                sitk.WriteImage(redacted_sitk, dcm_out)
                del redacted_sitk

                for line_id, data in pii_report.items():
                    if data.get("filtered"):
                        action = f"skipped ({data.get('filter_reason', '')})"
                    elif data["has_pii"]:
                        action = "redacted"
                    else:
                        action = "kept"

                    results.append({
                        "source_file": rel_path,
                        "accession_nbr": accession_nbr,
                        "person_id": person_id,
                        "line_index": int(line_id.split("_")[1]) if "_" in line_id else 0,
                        "text": data.get("text", ""),
                        "bounding_box": json.dumps(data.get("bounding_box", {})),
                        "has_pii": data.get("has_pii", False),
                        "action": action,
                        "pii_details": json.dumps(data.get("pii_details", [])),
                        "filter_reason": data.get("filter_reason") or "",
                        "processed_at": processed_at,
                        "output_path": dcm_out,
                        "error": None,
                    })

                if not pii_report:
                    results.append({
                        "source_file": rel_path,
                        "accession_nbr": accession_nbr,
                        "person_id": person_id,
                        "line_index": 0,
                        "text": "",
                        "bounding_box": "{}",
                        "has_pii": False,
                        "action": "no_text_detected",
                        "pii_details": "[]",
                        "filter_reason": "",
                        "processed_at": processed_at,
                        "output_path": dcm_out,
                        "error": None,
                    })

            except Exception as e:
                results.append({
                    "source_file": rel_path,
                    "accession_nbr": accession_nbr,
                    "person_id": person_id,
                    "line_index": 0,
                    "text": "",
                    "bounding_box": "{}",
                    "has_pii": False,
                    "action": "error",
                    "pii_details": "[]",
                    "filter_reason": "",
                    "processed_at": processed_at,
                    "output_path": None,
                    "error": str(e),
                })

            gc.collect()

        if results:
            yield pd.DataFrame(results)
        else:
            yield pd.DataFrame(columns=[
                "source_file", "accession_nbr", "person_id", "line_index",
                "text", "bounding_box", "has_pii", "action",
                "pii_details", "filter_reason", "processed_at",
                "output_path", "error",
            ])

    del ocr_engine
    gc.collect()

### Cell 9: Execute & Collect Results

In [0]:
dbutils.fs.mkdirs(OUTPUT_FOLDER)

print(f"Pass 2: starting distributed OCR across {num_gpu_partitions} GPU partitions (max_workers={MAX_WORKERS})...")
result_df = work_df.mapInPandas(process_batch, schema=_OCR_LOG_SCHEMA)

result_pdf = result_df.toPandas()
print(f"Processing complete: {len(result_pdf)} log rows collected")

### Cell 10: Save OCR Log to Delta

In [0]:
log_columns = [
    "source_file", "accession_nbr", "person_id", "line_index",
    "text", "bounding_box", "has_pii", "action",
    "pii_details", "filter_reason", "processed_at",
]

if not result_pdf.empty:
    catalog, schema_name, _ = OCR_LOG_TABLE.split(".")
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema_name}")

    log_schema = StructType([
        StructField("source_file", StringType(), True),
        StructField("accession_nbr", StringType(), True),
        StructField("person_id", StringType(), True),
        StructField("line_index", IntegerType(), True),
        StructField("text", StringType(), True),
        StructField("bounding_box", StringType(), True),
        StructField("has_pii", BooleanType(), True),
        StructField("action", StringType(), True),
        StructField("pii_details", StringType(), True),
        StructField("filter_reason", StringType(), True),
        StructField("processed_at", StringType(), True),
    ])

    log_df = spark.createDataFrame(result_pdf[log_columns], schema=log_schema)
    log_df.write.mode("append").option("mergeSchema", "true").saveAsTable(OCR_LOG_TABLE)
    print(f"Saved {len(result_pdf)} rows to {OCR_LOG_TABLE}")
else:
    print("No OCR log rows to save.")

### Cell 11: Summary

In [0]:
total_found = len(dicom_paths)
error_count = int(result_pdf["error"].notna().sum()) if not result_pdf.empty else 0
unique_files_processed = result_pdf["source_file"].nunique() if not result_pdf.empty else 0
text_regions = len(result_pdf[result_pdf["action"].isin(["redacted", "kept", "no_text_detected"])]) if not result_pdf.empty else 0
redacted_regions = int((result_pdf["action"] == "redacted").sum()) if not result_pdf.empty else 0

print("\n" + "=" * 60)
print("ANONYMIZATION SUMMARY (GPU-distributed, two-pass V2)")
print("=" * 60)
print(f"  Input folder:      {INPUT_FOLDER}")
print(f"  Output folder:     {OUTPUT_FOLDER}")
print(f"  Total files found: {total_found}")
print(f"  Readable:          {readable_count}")
print(f"  Skipped:           {skipped_count}")
print(f"  Multi-frame:       {multiframe_count}")
print(f"  Linked (selective):{linked_count}")
print(f"  Unlinked (blanket):{unlinked_count}")
print(f"  Files processed:   {unique_files_processed}")
print(f"  Errors:            {error_count}")
print(f"  Text regions:      {text_regions}")
print(f"  Redacted regions:  {redacted_regions}")
print(f"  OCR log table:     {OCR_LOG_TABLE}")
print(f"  GPU partitions:    {num_gpu_partitions}")
print(f"  Max workers:       {MAX_WORKERS}")
print(f"  Patients in batch: {len(matched_pids)}")
print("=" * 60)